# alternance-extractor -- QLoRA fine-tune (Qwen2.5-1.5B-Instruct)

**Status: ready to run.** `data/train/train.jsonl` exists (541 rows), has been cleaned up in
three rounds (duration-range backfills, intérim contract_type fix, and a full
required_skills/nice_to_have_skills re-labelling pass -- see `LABELLING-NOTES.md`), and
`MAX_SEQ_LEN` below has been calibrated against its real token-length distribution.

**Claim being tested**: a QLoRA fine-tune of Qwen2.5-1.5B-Instruct on Groq-labelled postings
can match Groq's Llama-3.3-70B-class baseline on structured extraction, at lower cost/latency.
This notebook does the fine-tuning; `notebooks/kaggle_benchmark.ipynb` scores both models with
`eval/score.py` against the hand-corrected `data/test/test.jsonl` (currently: Groq baseline
macro_f1=0.891, exact_match_rate=34.0%).

**Inputs this notebook expects** (upload as a Kaggle Dataset, or clone the repo if it's
public and `data/train/train.jsonl` has been committed):
- `schema/posting.py`, `label/prompt.py` -- reused so the fine-tuning prompt can never drift
  from the extraction schema or the Groq-labelling prompt.
- `data/train/train.jsonl` -- Groq-labelled training postings.

**Before running on Kaggle: push local commits to GitHub first** -- the repo-access cell below
clones from `https://github.com/mmattar18/alternance-extractor.git`, so any commits still only
local won't be visible to the Kaggle notebook.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets

## Repo access

Clones the repo so `schema.posting` and `label.prompt` can be imported directly, instead of
hand-duplicating the schema/prompt here and risking drift. If you've instead attached the repo
as a Kaggle Dataset input, skip the clone and just set `REPO_ROOT` to that input path.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/mmattar18/alternance-extractor.git"
REPO_ROOT = Path("/kaggle/working/alternance-extractor")

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

from label.prompt import SYSTEM_PROMPT  # noqa: E402  -- same schema/rules text used for Groq labelling

## Config

`MAX_SEQ_LEN` must comfortably cover `SYSTEM_PROMPT` + the longest `raw_text` + the JSON
target. Measured against the real `data/train/train.jsonl` (541 rows, `SYSTEM_PROMPT` alone is
1566 tokens): min=1755, p50=2174, p90=2594, p99=2925, max=3093 tokens. **The original 2048
default silently truncated 68.8% of training examples** -- almost always cutting into or
past the assistant's JSON target, which would have corrupted the fine-tune without any visible
error. Set to 3584 (comfortable headroom above the measured max). Re-check this if train.jsonl
grows or postings get meaningfully longer.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
TRAIN_PATH = REPO_ROOT / "data" / "train" / "train.jsonl"
OUTPUT_DIR = "/kaggle/working/qlora-adapter"
MAX_SEQ_LEN = 3584
NUM_EPOCHS = 1  # temporarily 1, not 3 -- see "Train" cell below for why
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
SEED = 42

## Load training data

Each row in `train.jsonl` is a labelled posting: original ingest fields plus Groq's `prediction`
(already schema-validated JSON, or `None` for the rows `select_test_set.py` dropped as invalid).
Training target is `json.dumps(prediction)` -- the exact string a correct extraction should
produce, so inference-time parsing (`schema.posting.parse_llm_json`) stays comparable to how
Groq's output was parsed during labelling.

Deliberately **not** including the few-shot examples from `label/prompt.py` here: those exist
to give Groq in-context examples it was never trained on. A fine-tuned model sees hundreds of
real examples via gradient updates instead, so baking the same 3 examples into every training
row would only burn context length for no benefit. `SYSTEM_PROMPT` (schema + extraction rules)
is reused as-is so the instructions stay identical to what labelled the data in the first place.

In [ ]:
records = [json.loads(l) for l in TRAIN_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
print(f"{len(records)} training postings loaded")

def to_chat_example(r):
    target = json.dumps(r["prediction"], ensure_ascii=False)
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": r["raw_text"]},
            {"role": "assistant", "content": target},
        ]
    }

dataset = Dataset.from_list([to_chat_example(r) for r in records])
dataset = dataset.train_test_split(test_size=0.05, seed=SEED)  # small held-out slice for eval_loss during training, distinct from data/test/test.jsonl
dataset

## Tokenizer, quantized base model, LoRA

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# fp16, not bf16: the T4 (Turing architecture) has no native bf16 tensor-core support --
# bf16 matmuls fall back to slower paths on this GPU. fp16 is fully tensor-core accelerated
# on T4. Kept consistent between the quantization compute dtype and training precision below.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Train

`packing=False`: each example is one posting -> one extraction, and packing multiple postings
into one sequence would let the model attend across unrelated postings, which we don't want for
this task. Costs some throughput (padding waste, since lengths range 1755-3093 against a
3584 cap) -- a deliberate correctness-over-speed tradeoff, not an oversight.

`max_length` and `packing` live on `SFTConfig` (not passed to `SFTTrainer` directly) as of
trl>=0.12 -- the API moved them there; passing them to `SFTTrainer.__init__` now raises
`TypeError: unexpected keyword argument`.

`loss_type="nll"`: trl's default (`"chunked_nll"`, a memory-saving variant) hits a real bug in
trl 1.10.0 -- it assumes `model.forward` is always a bound method with a `__func__` attribute,
which doesn't hold for this LoRA+4-bit model wrapping (`AttributeError: 'functools.partial'
object has no attribute '__func__'`). `"nll"` is the standard, mathematically equivalent loss
without the chunked optimization -- skips the buggy code path entirely.

`PER_DEVICE_BATCH_SIZE = 2`, `GRAD_ACCUMULATION_STEPS = 8` (effective batch size still 16):
the tradeoff for avoiding chunked_nll's bug is losing its memory savings too -- unchunked
`"nll"` materializes the full `[batch * seq_len, vocab_size]` logits tensor at once (Qwen's
vocab is ~152k), which at batch=8 tried to allocate 14GB by itself and OOM'd on the T4's
14.56GB usable capacity. Batch=2 keeps that tensor small enough to fit alongside the rest of
what's resident (model, optimizer state, activations).

`per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE`: **the actual full training run (batch=2)
completed an entire epoch successfully in ~2h11m, then OOM'd during the epoch-end eval pass.**
`per_device_eval_batch_size` was never set, so it silently defaulted to Hugging Face's default
of 8 -- the exact same full-vocab-logits OOM as before, just triggered by evaluation instead of
training. Pin it to match the train batch size.

`fp16=True` (was `bf16=True`): ~2h11m for one epoch (257 steps) is slow for a 1.5B model --
about 30s/step. The T4 (Turing) has no native bf16 tensor-core support, so bf16 ops were
likely running through a slower fallback path the whole time. Switched to fp16, which T4 fully
accelerates, and made the BitsAndBytesConfig compute dtype match (see model-loading cell above).

`NUM_EPOCHS=1` for now, not the eventual 3: five attempts and ~6 hours of wall-clock time in,
this run prioritizes getting one COMPLETE successful pass through the whole pipeline (train ->
save adapter -> feed into kaggle_benchmark.ipynb) before spending another several hours on the
full 3-epoch run. Bump back to 3 once the fp16 speedup is measured and the end-to-end chain is
confirmed working.

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_SEQ_LEN,
    packing=False,
    loss_type="nll",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

trainer.train()

## Save adapter

Saves the LoRA adapter only (small). Merge into the base model at inference time in
`notebooks/kaggle_benchmark.ipynb`, or with `model.merge_and_unload()` here if a single merged
checkpoint is more convenient for wherever benchmarking runs.

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

## Next step

Run `notebooks/kaggle_benchmark.ipynb` (not built yet) to generate predictions from this
adapter and the Groq baseline over `data/test/test.jsonl`, then score both with
`eval/score.py` -- that produces the three comparison numbers the README's claim needs:
field-level F1, exact-match rate, and JSON-validity rate, at whatever cost/latency each model
actually measured at.